In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install segmentation-models-pytorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.2 MB/s eta 0:00:00


In [3]:
import os



In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [9]:
import os
import cv2
import numpy as np
import torch
import segmentation_models_pytorch as smp

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

# =====================================================
# PATHS
# =====================================================

CHECKPOINT_DIR = "/content/drive/MyDrive/GLOC/checkpoints"

MODELS = [
    "baseline_best_unet.pth",
    "best_unet.pth",
    "pseudo_retrained_unet.pth",
    "bcp_best.pth"
]

# =====================================================
# DATASET
# =====================================================

class ValidationDataset(Dataset):

    def __init__(self, image_paths, mask_paths):

        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(
            self.mask_paths[idx],
            cv2.IMREAD_GRAYSCALE
        )

        image = image.astype(np.float32) / 255.0
        mask = (mask > 0).astype(np.float32)

        image = np.transpose(image, (2, 0, 1))

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        )

# =====================================================
# BUILD FILE LISTS
# =====================================================

IMAGE_ROOT = "/content/drive/MyDrive/GLOC/validation/images"
MASK_ROOT  = "/content/drive/MyDrive/GLOC/validation/labels"

image_paths = []
mask_paths = []

for cls in sorted(os.listdir(IMAGE_ROOT)):

    img_dir = os.path.join(
        IMAGE_ROOT,
        cls
    )

    mask_dir = os.path.join(
        MASK_ROOT,
        cls
    )

    if not os.path.isdir(img_dir):
        continue

    for file in os.listdir(img_dir):

        img_path = os.path.join(
            img_dir,
            file
        )

        mask_path = os.path.join(
            mask_dir,
            file
        )

        if os.path.exists(mask_path):

            image_paths.append(
                img_path
            )

            mask_paths.append(
                mask_path
            )

print(
    "Total validation images:",
    len(image_paths)
)

dataset = ValidationDataset(
    image_paths,
    mask_paths
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False
)

# =====================================================
# DEVICE
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# =====================================================
# TTA
# =====================================================

def predict_tta(model, images):

    pred1 = torch.sigmoid(model(images))

    hflip = torch.flip(images, dims=[3])

    pred2 = torch.sigmoid(model(hflip))
    pred2 = torch.flip(pred2, dims=[3])

    vflip = torch.flip(images, dims=[2])

    pred3 = torch.sigmoid(model(vflip))
    pred3 = torch.flip(pred3, dims=[2])

    return (pred1 + pred2 + pred3) / 3.0

# =====================================================
# EVALUATE FUNCTION
# =====================================================

def evaluate_checkpoint(checkpoint_path):

    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=3,
        classes=1
    ).to(device)

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device
        )
    )

    model.eval()

    total_tp = 0
    total_fp = 0
    total_fn = 0

    with torch.no_grad():

        for images, masks in tqdm(
            loader,
            leave=False
        ):

            images = images.to(device)

            probs = predict_tta(
                model,
                images
            )

            preds = (
                probs > 0.5
            ).float().cpu().numpy()

            masks = masks.numpy()

            for pred, gt in zip(
                preds,
                masks
            ):

                pred = pred.squeeze().astype(np.uint8)
                gt = gt.squeeze().astype(np.uint8)

                tp = np.logical_and(
                    pred == 1,
                    gt == 1
                ).sum()

                fp = np.logical_and(
                    pred == 1,
                    gt == 0
                ).sum()

                fn = np.logical_and(
                    pred == 0,
                    gt == 1
                ).sum()

                total_tp += tp
                total_fp += fp
                total_fn += fn

    iou = (
        total_tp /
        (
            total_tp +
            total_fp +
            total_fn +
            1e-8
        )
    )

    dice = (
        2 * total_tp /
        (
            2 * total_tp +
            total_fp +
            total_fn +
            1e-8
        )
    )

    return iou, dice

# =====================================================
# TEST ALL MODELS
# =====================================================

print("\n")
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

results = []

for model_name in MODELS:

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        model_name
    )

    print(f"\nEvaluating: {model_name}")

    iou, dice = evaluate_checkpoint(
        checkpoint_path
    )

    results.append(
        (model_name, iou, dice)
    )

    print(
        f"IoU  = {iou:.4f}"
    )

    print(
        f"Dice = {dice:.4f}"
    )

print("\n")
print("=" * 60)
print("FINAL RANKING")
print("=" * 60)

results.sort(
    key=lambda x: x[1],
    reverse=True
)

for rank, (name, iou, dice) in enumerate(results, start=1):

    print(
        f"{rank}. "
        f"{name:<28} "
        f"IoU={iou:.4f} "
        f"Dice={dice:.4f}"
    )

Total validation images: 2220


MODEL COMPARISON

Evaluating: baseline_best_unet.pth


IoU  = 0.4633
Dice = 0.6332

Evaluating: best_unet.pth


IoU  = 0.3318
Dice = 0.4982

Evaluating: pseudo_retrained_unet.pth


IoU  = 0.5079
Dice = 0.6736

Evaluating: bcp_best.pth


IoU  = 0.4913
Dice = 0.6589


FINAL RANKING
1. pseudo_retrained_unet.pth    IoU=0.5079 Dice=0.6736
2. bcp_best.pth                 IoU=0.4913 Dice=0.6589
3. baseline_best_unet.pth       IoU=0.4633 Dice=0.6332
4. best_unet.pth                IoU=0.3318 Dice=0.4982


In [11]:
IMAGE_ROOT = "/content/drive/MyDrive/GLOC/validation/images"
MASK_ROOT  = "/content/drive/MyDrive/GLOC/validation/labels"

image_paths = []
mask_paths = []

for cls in sorted(os.listdir(IMAGE_ROOT)):

    img_dir = os.path.join(
        IMAGE_ROOT,
        cls
    )

    mask_dir = os.path.join(
        MASK_ROOT,
        cls
    )

    if not os.path.isdir(img_dir):
        continue

    for file in os.listdir(img_dir):

        img_path = os.path.join(
            img_dir,
            file
        )

        mask_path = os.path.join(
            mask_dir,
            file
        )

        if os.path.exists(mask_path):

            image_paths.append(
                img_path
            )

            mask_paths.append(
                mask_path
            )

print(
    "Total validation images:",
    len(image_paths)
)

Total validation images: 2220


In [15]:
import os
import cv2
import numpy as np
import torch
import segmentation_models_pytorch as smp

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    cohen_kappa_score,
    jaccard_score
)

# =====================================================
# PATHS
# =====================================================
BASE_DIR = "/content/drive/MyDrive/GLOC"
IMAGE_DIR = os.path.join(BASE_DIR, "validation", "images")
MASK_DIR  = os.path.join(BASE_DIR, "validation", "labels")
CHECKPOINT = os.path.join(BASE_DIR, "checkpoints", "pseudo_retrained_unet.pth")

SAVE_PREDICTIONS = True
SAVE_DIR = "/content/drive/MyDrive/GLOC/predictions"

if SAVE_PREDICTIONS:
    os.makedirs(SAVE_DIR, exist_ok=True)

# =====================================================
# DEVICE
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [13]:
# =====================================================
# DATASET
# =====================================================

class ValidationDataset(Dataset):

    def __init__(
        self,
        image_paths,
        mask_paths
    ):

        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):

        return len(
            self.image_paths
        )

    def __getitem__(
        self,
        idx
    ):

        image = cv2.imread(
            self.image_paths[idx]
        )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        mask = cv2.imread(
            self.mask_paths[idx],
            cv2.IMREAD_GRAYSCALE
        )

        image = image.astype(
            np.float32
        ) / 255.0

        mask = (
            mask > 0
        ).astype(np.float32)

        image = np.transpose(
            image,
            (2,0,1)
        )

        return (
            torch.tensor(
                image,
                dtype=torch.float32
            ),
            torch.tensor(
                mask,
                dtype=torch.float32
            ).unsqueeze(0),
            os.path.basename(
                self.image_paths[idx]
            )
        )

# =====================================================
# DATALOADER
# =====================================================

dataset = ValidationDataset(
    image_paths,
    mask_paths
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Validation Images:", len(dataset))

Validation Images: 2220


In [16]:
# =====================================================
# MODEL
# =====================================================

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

model.load_state_dict(
    torch.load(
        CHECKPOINT,
        map_location=device
    )
)

model.eval()

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [17]:
# =====================================================
# TTA
# =====================================================

def predict_tta(model, images):

    pred1 = torch.sigmoid(
        model(images)
    )

    hflip = torch.flip(
        images,
        dims=[3]
    )

    pred2 = torch.sigmoid(
        model(hflip)
    )

    pred2 = torch.flip(
        pred2,
        dims=[3]
    )

    vflip = torch.flip(
        images,
        dims=[2]
    )

    pred3 = torch.sigmoid(
        model(vflip)
    )

    pred3 = torch.flip(
        pred3,
        dims=[2]
    )

    return (
        pred1 +
        pred2 +
        pred3
    ) / 3.0

In [22]:
import shutil

all_metrics = []

total_tp = 0
total_fp = 0
total_fn = 0
total_tn = 0

with torch.no_grad():

    img_counter = 0

    for images, masks, filenames in tqdm(loader):

        images = images.to(device)

        gt_masks = masks.numpy()

        preds = predict_tta(
            model,
            images
        )

        preds = (
            preds > 0.5
        ).float().cpu().numpy()

        for pred, gt, name in zip(
            preds,
            gt_masks,
            filenames
        ):

            pred = pred.squeeze().astype(np.uint8)
            gt   = gt.squeeze().astype(np.uint8)

            # =====================================================
            # SAVE PREDICTIONS + ORIGINAL IMAGES
            # =====================================================

            if SAVE_PREDICTIONS:

                # Save predicted mask
                cv2.imwrite(
                    os.path.join(
                        MASK_DIR,
                        name
                    ),
                    pred * 255
                )

                # Save corresponding original image
                original_img_path = image_paths[img_counter]

                shutil.copy(
                    original_img_path,
                    os.path.join(
                        IMAGE_DIR,
                        name
                    )
                )

            img_counter += 1

            # =====================================================
            # METRICS
            # =====================================================

            pred_flat = pred.flatten()
            gt_flat = gt.flatten()

            all_metrics.append({

                "IoU":
                    jaccard_score(
                        gt_flat,
                        pred_flat,
                        zero_division=0
                    ),

                "Dice":
                    f1_score(
                        gt_flat,
                        pred_flat,
                        zero_division=0
                    ),

                "Precision":
                    precision_score(
                        gt_flat,
                        pred_flat,
                        zero_division=0
                    ),

                "Recall":
                    recall_score(
                        gt_flat,
                        pred_flat,
                        zero_division=0
                    ),

                "Accuracy":
                    accuracy_score(
                        gt_flat,
                        pred_flat
                    ),

                "Kappa":
                    cohen_kappa_score(
                        gt_flat,
                        pred_flat
                    )
            })

            tp = np.logical_and(
                pred == 1,
                gt == 1
            ).sum()

            fp = np.logical_and(
                pred == 1,
                gt == 0
            ).sum()

            fn = np.logical_and(
                pred == 0,
                gt == 1
            ).sum()

            tn = np.logical_and(
                pred == 0,
                gt == 0
            ).sum()

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_tn += tn

100%|██████████| 278/278 [07:42<00:00,  1.66s/it]


In [23]:
# =====================================================
# DATASET-LEVEL METRICS
# =====================================================

dataset_iou = (
    total_tp /
    (
        total_tp +
        total_fp +
        total_fn +
        1e-8
    )
)

dataset_dice = (
    2 * total_tp /
    (
        2 * total_tp +
        total_fp +
        total_fn +
        1e-8
    )
)

dataset_precision = (
    total_tp /
    (
        total_tp +
        total_fp +
        1e-8
    )
)

dataset_recall = (
    total_tp /
    (
        total_tp +
        total_fn +
        1e-8
    )
)

dataset_accuracy = (
    total_tp + total_tn
) / (
    total_tp +
    total_tn +
    total_fp +
    total_fn +
    1e-8
)

# Cohen's Kappa

total_pixels = (
    total_tp +
    total_tn +
    total_fp +
    total_fn
)

po = dataset_accuracy

pe = (
    (
        (total_tp + total_fp) *
        (total_tp + total_fn)
    )
    +
    (
        (total_fn + total_tn) *
        (total_fp + total_tn)
    )
) / (
    (total_pixels ** 2) + 1e-8
)

dataset_kappa = (
    (po - pe) /
    (1 - pe + 1e-8)
)

print("\n")
print("=" * 60)
print("DATASET-LEVEL METRICS")
print("=" * 60)

print(
    f"{'Intersection over Union (IoU)':<35}: "
    f"{dataset_iou:.4f}"
)

print(
    f"{'F1 Score (Dice Score)':<35}: "
    f"{dataset_dice:.4f}"
)

print(
    f"{'Precision':<35}: "
    f"{dataset_precision:.4f}"
)

print(
    f"{'Recall':<35}: "
    f"{dataset_recall:.4f}"
)

print(
    f"{'Accuracy':<35}: "
    f"{dataset_accuracy:.4f}"
)

print(
    f"{'Kappa':<35}: "
    f"{dataset_kappa:.4f}"
)



DATASET-LEVEL METRICS
Intersection over Union (IoU)      : 0.5079
F1 Score (Dice Score)              : 0.6736
Precision                          : 0.7112
Recall                             : 0.6398
Accuracy                           : 0.9863
Kappa                              : 0.6666
